## Section 0 - Setup & Imports


In [ ]:
%pip install numpy scipy librosa soundfile matplotlib Pillow pydub ffmpeg-python pandas ipython

In [ ]:
# -- Setup environment in Colab --
import os

if not os.path.exists("assets"):
    print("Fetching assets from GitHub...")
    !git clone --quiet https://github.com/Monczak/audio-steganography.git /tmp/audio-steg
    !mv /tmp/audio-steg/assets .
    !rm -rf /tmp/audio-steg
    print("Assets mounted and ready!")

In [ ]:
import io
import wave
import struct
import subprocess
import tempfile
import warnings
import numpy as np
import scipy.io.wavfile as wav
import librosa
import librosa.display
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from IPython.display import Audio, display, HTML

warnings.filterwarnings("ignore")

SR = 44100
ASSETS_DIR = Path("assets")
AUDIO_DIR = ASSETS_DIR / "audio"
IMG_DIR = ASSETS_DIR / "img"

In [ ]:
# -- Helper Functions: Metrics --

def snr(original, modified):
    """Signal-to-Noise Ratio in dB."""
    original = np.asarray(original, dtype=np.float64)
    modified = np.asarray(modified, dtype=np.float64)
    noise = original - modified
    power_signal = np.sum(original ** 2)
    power_noise = np.sum(noise ** 2)
    if power_noise == 0:
        return float("inf")
    return 10 * np.log10(power_signal / power_noise)

def psnr(original, modified):
    """Peak Signal-to-Noise Ratio in dB."""
    original = np.asarray(original, dtype=np.float64)
    modified = np.asarray(modified, dtype=np.float64)
    noise = original - modified
    mse = np.mean(noise ** 2)
    if mse == 0:
        return float("inf")
    peak = np.max(np.abs(original))
    return 10 * np.log10(peak ** 2 / mse)

def ber(original_bytes, recovered_bytes):
    """Bit Error Rate: fraction of mismatched bits (now unpacks bytestrings)."""
    orig_arr = np.frombuffer(original_bytes, dtype=np.uint8)
    recov_arr = np.frombuffer(recovered_bytes, dtype=np.uint8)
    
    orig_bits = np.unpackbits(orig_arr) if len(orig_arr) > 0 else np.array([])
    recov_bits = np.unpackbits(recov_arr) if len(recov_arr) > 0 else np.array([])
    
    length = min(len(orig_bits), len(recov_bits))
    if length == 0:
        return 1.0
        
    errors = np.sum(orig_bits[:length] != recov_bits[:length])
    return errors / length

def capacity_bits(n_samples, bits_per_sample=1):
    """Maximum payload capacity in bits (excluding header)."""
    return n_samples * bits_per_sample

In [ ]:
# -- Helper Functions: Data Conversion --

def load_audio(file_path):
    """Load an audio file and ensures it is converted to mono."""
    audio, sr = sf.read(file_path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    return audio, sr

def text_to_bytes(text):
    """Convert a string directly to a bytestring."""
    return text.encode("utf-8")

def bytes_to_text(b):
    """Convert a bytestring back to a string."""
    return b.decode("utf-8", errors="replace")

In [ ]:
# -- Helper Functions: Visualization & Audio --

def plot_spectrogram(audio, sr, title="Spectrogram", ax=None):
    """Display a mel spectrogram."""
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(12, 4))
    S = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128)
    S_dB = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_dB, sr=sr, x_axis="time", y_axis="mel", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency (Hz)")

def plot_waveform(audio, sr, title="Waveform", ax=None):
    """Display the time-domain waveform."""
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(12, 3))
    t = np.arange(len(audio)) / sr
    ax.plot(t, audio, linewidth=0.5)
    ax.set_title(title)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")
    ax.set_xlim(0, t[-1])

def mp3_roundtrip(input_wav, bitrate="128k"):
    """WAV -> MP3 -> WAV round-trip via ffmpeg. Returns path to recovered WAV."""
    mp3_path = input_wav.replace(".wav", "_compressed.mp3")
    recovered_path = input_wav.replace(".wav", "_recovered.wav")
    subprocess.run(["ffmpeg", "-y", "-i", input_wav, "-b:a", bitrate, mp3_path], capture_output=True)
    subprocess.run(["ffmpeg", "-y", "-i", mp3_path, recovered_path], capture_output=True)
    return recovered_path

In [ ]:
# -- High-level Evaluation Helpers --

def display_audio_comparison(cover_audio, stego_audio, sr, title="Audio Comparison"):
    """Displays spectrograms and playable audio for cover vs stego."""
    print(f"\n{"="*20} {title} {"="*20}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    plot_spectrogram(cover_audio, sr, title="Original Spectrogram", ax=axes[0])
    plot_spectrogram(stego_audio, sr, title="Stego Spectrogram", ax=axes[1])
    plt.tight_layout()
    plt.show()
    
    print("Original Audio:")
    display(Audio(data=cover_audio, rate=sr))
    print("Stego Audio:")
    display(Audio(data=stego_audio, rate=sr))

## Section 1 - LSB Steganography

The simplest steganographic technique: replace the least significant bit(s) of each audio sample with payload data. Conceptually trivial, but it establishes the encode/decode loop, metrics vocabulary, and steganalysis baseline that every subsequent section builds on.

**Key trade-off:** More bits per sample = higher capacity but lower SNR and easier detection.


### 1.1 - LSB Encode & Decode (1-bit, 2-bit, 3-bit variants)


In [ ]:
def lsb_encode(audio_data, payload_bytes, n_bits=1):
    original_shape = audio_data.shape
    # Flatten array to seamlessly process stereo/multi-channel samples
    samples = np.int16(np.asarray(audio_data).flatten() * 32767)
    
    # Header: 32 bits (4 bytes) representing the payload length in bytes
    header = len(payload_bytes).to_bytes(4, byteorder="big")
    full_payload = header + payload_bytes
    
    total_bits = len(full_payload) * 8
    samples_needed = (total_bits + n_bits - 1) // n_bits
    
    if samples_needed > len(samples):
        raise ValueError(f"Payload too large: need {samples_needed} samples, have {len(samples)}")
        
    mask = ~((1 << n_bits) - 1) & 0xFFFF
    bit_idx = 0
    stego_samples = samples.copy()
    
    for i in range(samples_needed):
        val = int(stego_samples[i]) & 0xFFFF
        val = val & mask
        
        chunk = 0
        for b in range(n_bits):
            if bit_idx < total_bits:
                byte_val = full_payload[bit_idx // 8]
                bit_val = (byte_val >> (7 - (bit_idx % 8))) & 1
                chunk = (chunk << 1) | bit_val
                bit_idx += 1
            else:
                chunk = chunk << 1
                
        val = val | chunk
        
        # Restore two's complement constraint for negative values
        if val >= 0x8000:
            val -= 0x10000
        stego_samples[i] = val
        
    # Reshape the flattened outcome back to it's original multi-channel setup
    return (stego_samples.astype(np.float64) / 32767.0).reshape(original_shape)


def lsb_decode(stego_audio, n_bits=1):
    # Flatten array to seamlessly process stereo/multi-channel samples
    samples = np.int16(np.asarray(stego_audio).flatten() * 32767)
    
    extracted_bytes = bytearray()
    current_byte = 0
    bits_in_current_byte = 0
    
    payload_len = 0
    header_parsed = False
    
    for i in range(len(samples)):
        val = int(samples[i]) & 0xFFFF
        
        for b in range(n_bits - 1, -1, -1):
            bit_val = (val >> b) & 1
            current_byte = (current_byte << 1) | bit_val
            bits_in_current_byte += 1
            
            if bits_in_current_byte == 8:
                extracted_bytes.append(current_byte)
                current_byte = 0
                bits_in_current_byte = 0
                
                # Setup payload length once the 4-byte header is assembled
                if not header_parsed and len(extracted_bytes) == 4:
                    payload_len = int.from_bytes(extracted_bytes, byteorder='big')
                    header_parsed = True
                    
                # Return successfully if we've received the entire expected payload
                if header_parsed and len(extracted_bytes) == 4 + payload_len:
                    return bytes(extracted_bytes[4:])
                    
    # Failsafe if payload ends up truncated by audio end
    return bytes(extracted_bytes[4:4 + payload_len]) if len(extracted_bytes) > 4 else b""

In [ ]:
def test_lsb_variant(cover_audio, sr, payload_bytes, n_bits):
    """Runs an LSB encode/decode test in-memory and returns metrics."""
    stego_audio = lsb_encode(cover_audio, payload_bytes, n_bits)
    recovered_bytes = lsb_decode(stego_audio, n_bits)
    
    error_rate = ber(payload_bytes, recovered_bytes)
    quality = snr(cover_audio, stego_audio)
    cap = capacity_bits(len(cover_audio), n_bits)
    
    return {
        "Variant": f"{n_bits}-bit LSB",
        "SNR (dB)": round(quality, 2),
        "BER": round(error_rate, 4),
        "Capacity (bits)": cap,
        "Recovered Text Preview": bytes_to_text(recovered_bytes)[:50]
    }, stego_audio

In [ ]:
cover_name = "gong1.wav" # @param {type:"string"}
secret_message = "Audio steganography is the art of hiding information within sound. " * 300 # @param {type:"string"}

cover_audio, cover_sr = load_audio(AUDIO_DIR / cover_name)

payload = text_to_bytes(secret_message)
print(f"Original message preview: {secret_message[:100]}...")
print(f"Payload size: {len(payload) * 8} bits ({len(payload)} bytes)\n")

LSB_BIT_COUNTS = [1, 2, 3, 4, 5, 6, 7, 8]
results = []

for n_bits in LSB_BIT_COUNTS:
    try:
        metrics, stego = test_lsb_variant(cover_audio, cover_sr, payload, n_bits)
        results.append(metrics)
        
        if n_bits == 8:
            display_audio_comparison(cover_audio, stego, cover_sr, title=f"{n_bits}-bit LSB Example")
            
    except ValueError as e:
        print(f"Skipping {n_bits}-bit LSB: {e}")

if results:
    display(pd.DataFrame(results))

### 1.2 - SNR vs Bits-per-Sample & Waveform Comparison


In [ ]:
# -- SNR vs bits-per-sample sweep --
PAYLOAD_SIZE = 100000

np.random.seed(42)
large_payload = os.urandom(PAYLOAD_SIZE // 8)
bit_counts = [1, 2, 3, 4, 5, 6, 7, 8]
snr_values = []
capacities = []
stego_sweep_audios = {}

results = []
for n_bits in bit_counts:
    stego_audio = lsb_encode(cover_audio, large_payload, n_bits=n_bits)
    stego_sweep_audios[n_bits] = stego_audio
    snr_val = snr(cover_audio, stego_audio)
    cap = capacity_bits(len(cover_audio), n_bits)
    snr_values.append(snr_val)
    capacities.append(cap)
    results.append({"Bits": n_bits, "SNR (dB)": round(snr_val, 1), "Capacity (bytes)": cap // 8})
display(pd.DataFrame(results))


In [ ]:
# -- Waveform and SNR Plot --
start_idx = 10000
end_idx = 10200

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(bit_counts, snr_values, color="steelblue", edgecolor="black")
axes[0].set_xlabel("Bits per Sample")
axes[0].set_ylabel("SNR (dB)")
axes[0].set_title("SNR vs Bits per Sample (LSB)")
axes[0].set_xticks(bit_counts)
for i, v in enumerate(snr_values):
    axes[0].text(bit_counts[i], v + 1, f"{v:.0f}", ha="center", fontsize=9)
axes[1].bar(bit_counts, [c / 1e6 for c in capacities], color="coral", edgecolor="black")
axes[1].set_xlabel("Bits per Sample")
axes[1].set_ylabel("Capacity (Mbit)")
axes[1].set_title("Payload Capacity vs Bits")
axes[1].set_xticks(bit_counts)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
t = np.arange(start_idx, end_idx) / cover_sr * 1000
axes[0].plot(t, cover_audio[start_idx:end_idx], "b-", linewidth=1, label="Original")
axes[0].set_title("Original Audio (zoomed)")
axes[0].set_ylabel("Amplitude")
for idx, n_bits in enumerate([1, 3]):
    stego = stego_sweep_audios[n_bits]
    diff = cover_audio[start_idx:end_idx] - stego[start_idx:end_idx]
    axes[idx+1].plot(t, cover_audio[start_idx:end_idx], "b-", linewidth=1, alpha=0.5, label="Original")
    axes[idx+1].plot(t, stego[start_idx:end_idx], "r--", linewidth=1, alpha=0.7, label=f"{n_bits}-bit LSB")
    axes[idx+1].set_title(f"{n_bits}-bit LSB vs Original (max diff = {np.max(np.abs(diff)):.6f})")
    axes[idx+1].set_ylabel("Amplitude")
    axes[idx+1].legend(loc="upper right")
axes[-1].set_xlabel("Time (ms)")
plt.tight_layout()
plt.show()


### 1.3 - Steganalysis: Chi-Squared LSB Distribution Test

In natural audio, the LSB of consecutive samples is weakly correlated. Embedding a payload can skew the distribution of LSBs. A chi-squared goodness-of-fit test can detect this shift.


In [ ]:
def lsb_steganalysis(audio_data, n_bits=1):
    """
    Chi-squared test on LSB distribution of audio array.
    Returns: chi2_stat, p_value, lsb_ratio
    """
    samples = np.int16(audio_data * 32767)
    lsbs = [int(s) & ((1 << n_bits) - 1) for s in samples]
    n_values = 1 << n_bits
    
    observed = np.zeros(n_values)
    for lsb in lsbs:
        observed[lsb] += 1
        
    expected = np.full(n_values, len(lsbs) / n_values)
    chi2_stat = np.sum((observed - expected) ** 2 / expected)
    
    from scipy.stats import chi2
    p_value = 1 - chi2.cdf(chi2_stat, df=n_values - 1)
    lsb_ratio = np.mean([b & 1 for b in lsbs])
    
    return chi2_stat, p_value, lsb_ratio

In [ ]:
# -- Run steganalysis on clean and stego files --
N_BITS_LIST = [1, 2, 3, 4, 5, 6, 7, 8]
PAYLOAD_SIZE = 100000

results = []
chi2_s, p_val, ratio = lsb_steganalysis(cover_audio)
results.append({
    "File": "Original (clean)",
    "Chi2 stat": round(chi2_s, 2),
    "p-value": p_val,
    "LSB=1 ratio": round(ratio, 4),
    "Verdict": "CLEAN" if p_val > 0.05 else "STEGO DETECTED"
})

text_message = "Audio steganography is the art of hiding information within sound. "
repeated_text = text_message * ((PAYLOAD_SIZE // 8) // len(text_message) + 1)
text_payload = text_to_bytes(repeated_text)[:PAYLOAD_SIZE // 8]

for n_bits in N_BITS_LIST:
    stego_audio = lsb_encode(cover_audio, text_payload, n_bits)
    chi2_s, p_val, ratio = lsb_steganalysis(stego_audio, n_bits)
    results.append({
        "File": f"Stego {n_bits}-bit LSB (text payload)",
        "Chi2 stat": round(chi2_s, 2),
        "p-value": p_val,
        "LSB=1 ratio": round(ratio, 4),
        "Verdict": "CLEAN" if p_val > 0.05 else "STEGO DETECTED"
    })

display(pd.DataFrame(results))

In [ ]:
# -- Visualize LSB histogram: clean vs stego --
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

samples_clean = np.int16(cover_audio * 32767)
lsbs_clean = [int(s) & 1 for s in samples_clean]

# Use the text payload from the previous cell instead of random bytes
stego_audio_1bit = lsb_encode(cover_audio, text_payload, n_bits=1)
samples_stego = np.int16(stego_audio_1bit * 32767)
lsbs_stego = [int(s) & 1 for s in samples_stego]

axes[0].bar([0, 1], [lsbs_clean.count(0), lsbs_clean.count(1)], color=["steelblue", "coral"], edgecolor="black")
axes[0].set_title(f"Clean Audio LSB Distribution\n(ratio = {np.mean(lsbs_clean):.4f})")
axes[0].set_xlabel("LSB Value")
axes[0].set_ylabel("Count")
axes[0].set_xticks([0, 1])

axes[1].bar([0, 1], [lsbs_stego.count(0), lsbs_stego.count(1)], color=["steelblue", "coral"], edgecolor="black")
axes[1].set_title(f"Stego Audio LSB Distribution\n(ratio = {np.mean(lsbs_stego):.4f})")
axes[1].set_xlabel("LSB Value")
axes[1].set_ylabel("Count")
axes[1].set_xticks([0, 1])

plt.tight_layout()
plt.show()

### 1.4 - Robustness Test: MP3 Round-Trip

LSB embedding modifies the least significant bits of PCM samples. MP3 compression uses lossy psychoacoustic modelling that completely reshuffles sample values - the LSBs are destroyed. This demonstrates *why* more sophisticated techniques exist.


In [ ]:
def mp3_roundtrip_variable(audio_data, sr, bitrate="128k"):
    """WAV -> MP3 -> WAV round-trip passing data through temp files."""
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as wav_in, \
        tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as mp3_out, \
        tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as wav_out:
        
        sf.write(wav_in.name, audio_data, sr, subtype="PCM_16")
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", wav_in.name, "-b:a", bitrate, mp3_out.name], capture_output=True)
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", mp3_out.name, wav_out.name], capture_output=True)
        
        recovered_audio, _ = sf.read(wav_out.name)
        
    os.unlink(wav_in.name)
    os.unlink(mp3_out.name)
    os.unlink(wav_out.name)
    
    return recovered_audio

In [ ]:
# -- MP3 Round-Trip Robustness Test --
BITRATES = ["320k", "192k", "128k", "64k"]
payload = text_to_bytes("This message will not survive MP3 compression.")
stego_audio = lsb_encode(cover_audio, payload, n_bits=1)

results = []
clean_recovered = lsb_decode(stego_audio, n_bits=1)
clean_ber = ber(payload, clean_recovered)
results.append({
    "Bitrate": "Before MP3",
    "BER": f"{clean_ber:.2%}",
    "Recovered Text": bytes_to_text(clean_recovered)[:50]
})

for bitrate in BITRATES:
    recovered_audio = mp3_roundtrip_variable(stego_audio, cover_sr, bitrate)
    try:
        recovered_bits = lsb_decode(recovered_audio, n_bits=1)
        error_rate = ber(payload, recovered_bits)
        recovered_text = bytes_to_text(recovered_bits)[:50]
    except Exception as e:
        error_rate = 1.0
        recovered_text = f"(extraction failed)"
    
    results.append({
        "Bitrate": bitrate,
        "BER": f"{error_rate:.2%}",
        "Recovered Text": recovered_text
    })

display(pd.DataFrame(results))
